Step 1: Install Required Libraries

In [1]:
!pip install transformers torch PyPDF2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.0 MB/s eta 0:00:00


Step 2: Upload Your PDF

In [3]:
from google.colab import files
uploaded = files.upload()   # select file3.pdf

Saving file3.pdf to file3 (1).pdf


Step 3: Extract Text from the PDF

In [4]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

context = extract_text_from_pdf("file3.pdf")
print(context[:1000])   # preview

Bangladesh
 
is
 
a
 
beautiful
 
country
 
in
 
South
 
Asia,
 
known
 
for
 
its
 
rich
 
cultural
 
heritage,
 
fertile
 
land,
 
and
 
vast
 
river
 
systems.
 
The
 
capital
 
city
 
of
 
Bangladesh
 
is
 
Dhaka.
 
Dhaka
 
is
 
not
 
only
 
the
 
political
 
center
 
but
 
also
 
the
 
economic
 
and
 
cultural
 
heart
 
of
 
the
 
nation.
 
It
 
is
 
one
 
of
 
the
 
most
 
densely
 
populated
 
cities
 
in
 
the
 
world
 
and
 
plays
 
a
 
major
 
role
 
in
 
the
 
country’s
 
development.
 
 
One
 
of
 
the
 
most
 
important
 
natural
 
features
 
of
 
Bangladesh
 
is
 
the
 
Padma
 
River,
 
which
 
is
 
often
 
referred
 
to
 
as
 
the
 
lifeline
 
of
 
the
 
country.
 
This
 
river
 
is
 
crucial
 
for
 
transportation,
 
agriculture,
 
fishing,
 
and
 
daily
 
life.
 
Many
 
people
 
depend
 
on
 
it
 
for
 
their
 
livelihoods,
 
and
 
it
 
supports
 
the
 
fertile
 
land
 
that
 
makes
 
Bangladesh
 
suitable
 
for
 
farming.
 
 
Bangladesh
 
gained
 
its
 
independence


Step 4: Load a Pre-trained BERT QA Model

We'll use bert-large-uncased-whole-word-masking-finetuned-squad

In [5]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="bert-large-uncased-whole-word-masking-finetuned-squad",
    tokenizer="bert-large-uncased-whole-word-masking-finetuned-squad"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-large-uncased-whole-word-masking-finetuned-squad
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Step 5: Handle Long PDFs (Chunking)

BERT has a 512-token limit. Splitting context into overlapping chunks and then pick the best answer

In [6]:
def chunk_text(text, chunk_size=2000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def answer_question(question, context):
    chunks = chunk_text(context)
    best = {"score": 0, "answer": ""}
    for chunk in chunks:
        try:
            result = qa_pipeline(question=question, context=chunk)
            if result["score"] > best["score"]:
                best = result
        except Exception:
            continue
    return best["answer"], best["score"]

Step 6: Ask the Questions

In [7]:
questions = [
    "What is the capital city of Bangladesh?",
    "Which river is considered the lifeline of Bangladesh?",
    "In which year did Bangladesh gain independence?",
    "What is the official language of Bangladesh?",
    "What is the name of the world's largest mangrove forest located in Bangladesh?",
    "Which currency is used in Bangladesh?",
    "What is the national animal of Bangladesh?"
]

for q in questions:
    ans, score = answer_question(q, context)
    print(f"Q: {q}\nA: {ans}  (confidence: {score:.2f})\n")

Q: What is the capital city of Bangladesh?
A: Dhaka  (confidence: 0.95)

Q: Which river is considered the lifeline of Bangladesh?
A: Padma
 
River  (confidence: 0.98)

Q: In which year did Bangladesh gain independence?
A: 1971  (confidence: 0.99)

Q: What is the official language of Bangladesh?
A: Bengali  (confidence: 0.92)

Q: What is the name of the world's largest mangrove forest located in Bangladesh?
A: Sundarbans  (confidence: 1.78)

Q: Which currency is used in Bangladesh?
A: Bangladeshi
 
Taka  (confidence: 0.81)

Q: What is the national animal of Bangladesh?
A: Royal
 
Bengal
 
Tiger  (confidence: 1.49)



Step 7: Interactive Mode

In [8]:
while True:
    q = input("Ask a question (or 'quit'): ")
    if q.lower() == "quit":
        break
    ans, score = answer_question(q, context)
    print(f"Answer: {ans} (confidence: {score:.2f})\n")

Ask a question (or 'quit'): What is the capital city of Bangladesh?
Answer: Dhaka (confidence: 0.95)

Ask a question (or 'quit'): quit
